# Preprocesado de texto v2
Features concretas de `Description` para el arbol: estadisticas simples, preguntas zero-shot (NLI) y embeddings multilingues (para `txt_score` dentro de cada fold).

In [ ]:
# Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API: armar ../input/<nombre>
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        for d in glob.glob("/kaggle/input/" + "*/" * prof):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(nombre)
_inputs = {"petfinder-adoption-prediction": "train/train.csv"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        if not os.path.exists(f"/tmp/lab/input/{n}"):
            os.symlink(_buscar(n, a), f"/tmp/lab/input/{n}")
    os.chdir("/tmp/lab/work")
D = "../input/petfinder-adoption-prediction"
OUT = "/kaggle/working"

In [ ]:
import re
import numpy as np
import pandas as pd
train = pd.read_csv(f"{D}/train/train.csv").set_index("PetID")
test = pd.read_csv(f"{D}/test/test.csv").set_index("PetID")
desc = pd.concat([train["Description"], test["Description"]]).fillna("").astype(str).str.strip()
assert desc.index.is_unique
ids_train, ids_test = train.index, test.index
print(len(ids_train), len(ids_test), "sin descripcion:", (desc == "").sum())

In [ ]:
# 1) estadisticas simples (sin modelo)
MALAYO = {"dan", "yang", "untuk", "ini", "itu", "boleh", "tidak", "saya", "dia", "ada", "sangat", "anak",
          "kucing", "anjing", "dengan", "nak", "jantan", "betina", "sudah", "sihat", "comel", "manja", "akan",
          "lagi", "sila", "hubungi", "percuma", "mahu", "kami", "dari", "pada", "dalam", "tak", "dah"}

def stats(t):
    letras = sum(c.isalpha() for c in t)
    palabras = re.findall(r"[A-Za-z]+", t)
    return {
        "txt_largo": len(t),
        "txt_palabras": len(t.split()),
        "txt_frac_chino": len(re.findall(r"[\u4e00-\u9fff]", t)) / len(t) if t else np.nan,
        "txt_frac_malayo": sum(w.lower() in MALAYO for w in palabras) / len(palabras) if palabras else np.nan,
        "txt_frac_mayus": sum(c.isupper() for c in t) / letras if letras else np.nan,
        "txt_exclamaciones": t.count("!"),
        "txt_telefono": int(bool(re.search(r"\d{2,4}[\s-]?\d{3,4}[\s-]?\d{3,4}", t))),
        "txt_precio": int(bool(re.search(r"\b(rm\s?\d+|fee|fees|ringgit)\b|\$\s?\d+", t, re.I))),
    }

st = pd.DataFrame([stats(t) for t in desc], index=desc.index)
st.loc[ids_train].to_parquet(f"{OUT}/txt_stats_train.parquet")
st.loc[ids_test].to_parquet(f"{OUT}/txt_stats_test.parquet")
st.describe().T

In [ ]:
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:
    (torch.ones(4, 4, device=dev) @ torch.ones(4, 4, device=dev)).sum().item()
except Exception as e:  # GPU que la version de torch ya no soporta
    print("cuda no anda, sigo en cpu:", e)
    dev = "cpu"
print("torch", torch.__version__, dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

In [ ]:
# 2) preguntas zero-shot: probabilidad de que la descripcion implique cada frase (modelo NLI)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
NLI = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0" if dev == "cuda" else "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tok = AutoTokenizer.from_pretrained(NLI)
nli = AutoModelForSequenceClassification.from_pretrained(NLI).to(dev).eval()
if dev == "cuda":
    nli = nli.half()
ENT = [i for l, i in nli.config.label2id.items() if l.lower().startswith("entail")][0]
print(NLI, nli.config.id2label, "entailment =", ENT)

PREGUNTAS = {
    "txt_urgente": "The owner needs to give this pet away urgently.",
    "txt_enfermo": "This pet is sick, injured or disabled.",
    "txt_entrenado": "This pet is toilet trained or house trained.",
    "txt_ninos": "This pet is good with children.",
    "txt_camada": "This pet is one of several puppies or kittens from the same litter.",
    "txt_cobra": "The adopter has to pay a fee or a price for this pet.",
}
con_texto = desc[desc != ""]
orden = con_texto.str.len().sort_values().index  # lotes de largo parecido: menos padding
zs = pd.DataFrame(np.nan, index=desc.index, columns=list(PREGUNTAS))
B = 64
with torch.no_grad():
    for col, hip in PREGUNTAS.items():
        res = []
        for k in range(0, len(orden), B):
            t = con_texto.loc[orden[k:k + B]].tolist()
            enc = tok(t, [hip] * len(t), truncation="only_first", max_length=256, padding=True, return_tensors="pt").to(dev)
            res.append(torch.softmax(nli(**enc).logits.float(), dim=-1)[:, ENT].cpu().numpy())
        zs.loc[orden, col] = np.concatenate(res)
        print(col, "listo", flush=True)
zs.loc[ids_train].to_parquet(f"{OUT}/txt_zs_train.parquet")
zs.loc[ids_test].to_parquet(f"{OUT}/txt_zs_test.parquet")
# control: las 3 descripciones con mas probabilidad en cada pregunta
for col in PREGUNTAS:
    print("==", col)
    for i in zs[col].nlargest(3).index:
        print("  ", round(zs.loc[i, col], 3), desc[i][:150].replace("\n", " "))
del nli
torch.cuda.empty_cache() if dev == "cuda" else None

In [ ]:
# 3) embeddings multilingues (768) para armar txt_score dentro de cada fold en el experimento
from transformers import AutoModel
EMB = "intfloat/multilingual-e5-base"
tok = AutoTokenizer.from_pretrained(EMB)
enc_model = AutoModel.from_pretrained(EMB).to(dev).eval()
if dev == "cuda":
    enc_model = enc_model.half()
vecs = []
with torch.no_grad():
    for k in range(0, len(orden), B):
        t = ["query: " + x for x in con_texto.loc[orden[k:k + B]]]
        enc = tok(t, truncation=True, max_length=512, padding=True, return_tensors="pt").to(dev)
        h = enc_model(**enc).last_hidden_state.float()
        m = enc["attention_mask"].unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1)
        vecs.append(torch.nn.functional.normalize(v, dim=-1).cpu().numpy())
emb = pd.DataFrame(np.nan, index=desc.index, columns=[f"txt_emb_{i}" for i in range(768)], dtype="float32")
emb.loc[orden] = np.concatenate(vecs).astype("float32")
emb.loc[ids_train].to_parquet(f"{OUT}/txt_emb_train.parquet")
emb.loc[ids_test].to_parquet(f"{OUT}/txt_emb_test.parquet")
print(emb.shape, "filas sin embedding:", emb.isna().all(axis=1).sum())